# LoRA Fine-Tuning on GPU — Google Colab

**Project:** Sycophancy & Bias Fine-Tuning  
**Model:** TinyLlama-1.1B-Chat-v1.0  
**Dataset:** Alpaca (tatsu-lab)

### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells top to bottom
3. At the end, download `training_output.zip` and unzip into your local `training/output/` folder
4. Resume locally from Phase 6: `python run_full_pipeline.py --from 6`

## 1. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Install dependencies

In [ ]:
!pip install -q transformers datasets peft trl accelerate

## 3. Configuration — edit these values if needed

In [ ]:
MODEL_NAME        = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
NUM_TRAIN_SAMPLES = 3000   # GPU can handle much more than CPU — increase to 5000 if time allows
NUM_VAL_SAMPLES   = 200
EPOCHS            = 3
BATCH_SIZE        = 8
GRAD_ACCUM        = 2      # effective batch = 16
LEARNING_RATE     = 2e-4
SAVE_STEPS        = 100    # checkpoint frequency — must match local setup
MAX_SEQ_LEN       = 512
OUTPUT_DIR        = '/content/training_output'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config set. Output dir:', OUTPUT_DIR)

## 4. Load and format Alpaca dataset

In [ ]:
from datasets import load_dataset

print('Loading Alpaca...')
alpaca = load_dataset('tatsu-lab/alpaca', split='train')
print(f'Full dataset: {len(alpaca)} samples')

total = NUM_TRAIN_SAMPLES + NUM_VAL_SAMPLES
subset = alpaca.shuffle(seed=42).select(range(min(total, len(alpaca))))
train_raw = subset.select(range(NUM_TRAIN_SAMPLES))
val_raw   = subset.select(range(NUM_TRAIN_SAMPLES, total))

print(f'Train: {len(train_raw)}  |  Val: {len(val_raw)}')

In [ ]:
def format_sample(sample):
    instruction = sample.get('instruction', '').strip()
    inp         = sample.get('input', '').strip()
    output      = sample.get('output', '').strip()
    user_msg    = f'{instruction}\n\nInput: {inp}' if inp else instruction
    text = (
        f'<|user|>\n{user_msg}</s>\n'
        f'<|assistant|>\n{output}</s>'
    )
    return {'text': text}

cols = train_raw.column_names
train_fmt = train_raw.map(format_sample, remove_columns=cols)
val_fmt   = val_raw.map(format_sample,   remove_columns=cols)

print('Sample formatted text:')
print(train_fmt[0]['text'][:300])

## 5. Load TinyLlama with LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train

In [ ]:
try:
    from trl import SFTTrainer, SFTConfig
    USE_SFT_CONFIG = True
except ImportError:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    USE_SFT_CONFIG = False

common_args = dict(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    logging_steps               = 10,
    eval_strategy               = 'steps',
    eval_steps                  = SAVE_STEPS,
    save_strategy               = 'steps',
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 20,
    load_best_model_at_end      = False,
    fp16                        = True,
    seed                        = 42,
    report_to                   = 'none',
)

if USE_SFT_CONFIG:
    training_args = SFTConfig(
        **common_args,
        max_seq_length     = MAX_SEQ_LEN,
        dataset_text_field = 'text',
        packing            = False,
    )
    trainer = SFTTrainer(
        model         = model,
        train_dataset = train_fmt,
        eval_dataset  = val_fmt,
        args          = training_args,
        tokenizer     = tokenizer,
    )
else:
    from transformers import TrainingArguments
    training_args = TrainingArguments(**common_args)
    trainer = SFTTrainer(
        model              = model,
        train_dataset      = train_fmt,
        eval_dataset       = val_fmt,
        args               = training_args,
        tokenizer          = tokenizer,
        dataset_text_field = 'text',
        max_seq_length     = MAX_SEQ_LEN,
    )

print('Starting training...')
trainer.train()

## 7. Save final model

In [ ]:
import json

final_path = os.path.join(OUTPUT_DIR, 'final_model')
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

config = {
    'model': MODEL_NAME, 'dataset': 'tatsu-lab/alpaca',
    'num_train_samples': NUM_TRAIN_SAMPLES, 'epochs': EPOCHS,
    'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
    'lora_r': 8, 'lora_alpha': 16, 'device': 'cuda'
}
with open(os.path.join(OUTPUT_DIR, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('Model saved to:', final_path)
print('Checkpoints in:', OUTPUT_DIR)
print('\nCheckpoints found:')
for item in sorted(os.listdir(OUTPUT_DIR)):
    print(' ', item)

## 8. Download checkpoints to your local machine

This zips the entire output directory and downloads it. Unzip it into your local `training/output/` folder, then run:
```bash
python run_full_pipeline.py --from 6
```

In [ ]:
import shutil
from google.colab import files

print('Zipping checkpoints...')
zip_path = shutil.make_archive('/content/training_output', 'zip', OUTPUT_DIR)
print(f'Zip created: {zip_path}')
print(f'Size: {os.path.getsize(zip_path) / 1e6:.1f} MB')

print('\nDownloading...')
files.download(zip_path)
print('Done! Unzip into your local training/output/ directory.')